In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import os
import numpy as np
import pandas as pd

from src.anonymization import mdav_c, rmdav_m

In [ ]:
# Input file
INPUT_PATH = "../data/case_embeddings.parquet"

# Output directory
OUTPUT_DIR = "../data/anonymized_outputs"

# Cluster sizes
K_VALUES = [5, 10, 25, 50, 100]

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
df = pd.read_parquet(INPUT_PATH)
print(df.shape)
df.head()

In [ ]:
embedding_columns = sorted(
    [c for c in df.columns if str(c).isdigit()],
    key=int,
)

X = df[embedding_columns].to_numpy(dtype=float)

print(X.shape)

In [ ]:
for k in K_VALUES:

    print(f"Running MDAV-C (k={k})...")

    X_mdav = mdav_c(X, k)

    output = df.copy()
    output[embedding_columns] = X_mdav

    output.to_parquet(
        f"{OUTPUT_DIR}/mdav_c_k{k}.parquet",
        index=False,
    )

In [ ]:
for k in K_VALUES:

    print(f"Running RMDAV-M (k={k})...")

    X_rmdav = rmdav_m(
        X,
        k,
        random_state=42,
    )

    output = df.copy()
    output[embedding_columns] = X_rmdav

    output.to_parquet(
        f"{OUTPUT_DIR}/rmdav_m_k{k}.parquet",
        index=False,
    )

In [ ]:
print("Finished.")

print(f"Saved outputs to: {OUTPUT_DIR}")